# Gold — the tables the article reads from

Five outputs: the local authority distribution, the gap expressed in people, the coverage timeline,
the publication register, and a facts table.

The facts table matters most. Every number that appears in the article is a named row in it, with
the source it came from and whether it was computed or read off a page. Nothing gets typed into
prose by hand.

Reads `data/silver/` and the pages Bronze captured. Requires `02_silver.ipynb` to have run.

## 1. Paths and inputs

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

BASE_DIR     = Path.home() / "projects" / "nhs-cervical-screening-pipeline"
BRONZE_DIR   = BASE_DIR / "data" / "bronze"
EVIDENCE_DIR = BRONZE_DIR / "evidence"
SILVER_DIR   = BASE_DIR / "data" / "silver"
GOLD_DIR     = BASE_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

coverage     = pd.read_parquet(SILVER_DIR / "coverage.parquet")
publications = pd.read_parquet(SILVER_DIR / "publications.parquet")

# Fixed reference points. The announcement date anchors every "how stale was it" figure so the
# article's numbers do not drift with the day the notebook happens to run.
ANNOUNCEMENT = pd.Timestamp("2026-08-25")
MAR_2024     = pd.Timestamp("2024-03-31")
TARGET_PCT   = 80.0
AS_AT        = pd.Timestamp("today").normalize()

facts = []
def fact(name, value, unit="", basis="computed", source="", note=""):
    facts.append(dict(name=name, value=value, unit=unit, basis=basis, source=source, note=note))
    return value

print(f"coverage {len(coverage):,} rows | publications {len(publications)} rows | as at {AS_AT.date()}")

## 2. The local authority distribution

The annual publication is the accredited source and the one the 68.8% headline comes from, so it is
the primary here. It reports 151 local authorities; the quarterly management information reports 153
because it lists City of London and Isles of Scilly separately rather than folding them into their
neighbours. Both counts are recorded — the difference changes which authority sits at the bottom of
any ranking.

In [ ]:
la = (coverage[(coverage.series == "annual")
               & (coverage.period_end == MAR_2024)
               & (coverage.geography_level == "Local Authority")
               & (coverage.age_band == "25-64")]
      [["org_code", "org_name", "eligible", "screened", "coverage_pct_computed"]]
      .rename(columns={"coverage_pct_computed": "coverage_pct"})
      .sort_values("coverage_pct")
      .reset_index(drop=True))

la["rank"] = la.coverage_pct.rank(ascending=False, method="min").astype(int)
la["gap_to_target_pp"] = TARGET_PCT - la.coverage_pct
la["at_or_above_target"] = la.coverage_pct >= TARGET_PCT

england = 100 * la.screened.sum() / la.eligible.sum()

fact("headline_coverage_pct", round(england, 4), "%", "computed", "annual_phof",
     "Summed across local authority numerators and denominators, not averaged")
fact("headline_coverage_published", 68.8, "%", "read from source", "annual_phof",
     "As printed on the publication page and quoted in the press release")
fact("eligible_25_64", int(la.eligible.sum()), "people", "computed", "annual_phof")
fact("screened_25_64", int(la.screened.sum()), "people", "computed", "annual_phof")
fact("target_pct", TARGET_PCT, "%", "read from source", "nhs_england_press_release")

fact("la_count_annual", len(la), "authorities", "computed", "annual_phof")
fact("la_at_or_above_target", int(la.at_or_above_target.sum()), "authorities", "computed", "annual_phof")
fact("la_coverage_min", round(la.coverage_pct.min(), 2), "%", "computed", "annual_phof")
fact("la_coverage_max", round(la.coverage_pct.max(), 2), "%", "computed", "annual_phof")
fact("la_coverage_median", round(la.coverage_pct.median(), 2), "%", "computed", "annual_phof")
fact("la_spread_pp", round(la.coverage_pct.max() - la.coverage_pct.min(), 2), "pp", "computed", "annual_phof")
fact("la_lowest_name", la.iloc[0].org_name, "", "computed", "annual_phof")
fact("la_highest_name", la.iloc[-1].org_name, "", "computed", "annual_phof")
fact("la_below_england", int((la.coverage_pct < england).sum()), "authorities", "computed", "annual_phof")

# the quarterly file's wider list, for the count that differs
la_mi = coverage[(coverage.series == "mi_quarterly") & (coverage.period_end == MAR_2024)
                 & (coverage.geography_level == "Local Authority") & (coverage.age_band == "25-64")]
fact("la_count_quarterly", int(la_mi.org_name.nunique()), "authorities", "computed", "mi_quarterly_la",
     "Lists City of London and Isles of Scilly separately")

print(f"{len(la)} authorities | {la.at_or_above_target.sum()} at or above {TARGET_PCT:.0f}% | "
      f"England {england:.4f}%")
print(f"range {la.coverage_pct.min():.2f} to {la.coverage_pct.max():.2f}, "
      f"median {la.coverage_pct.median():.2f}")
print()
print("lowest five"); print(la.head(5)[["org_name", "coverage_pct"]].to_string(index=False))
print("highest five"); print(la.tail(5)[["org_name", "coverage_pct"]].to_string(index=False))

## 3. The gap, in people

`screens_to_target` is the publisher's own column, not my arithmetic: how many additional screens
each authority needs to reach 80%. Only the quarterly management information carries it.

Summed, it gives the size of the gap in the same unit as the intervention — screens against kits.

In [ ]:
gap = coverage[(coverage.series == "mi_quarterly") & (coverage.period_end == MAR_2024)
               & (coverage.geography_level == "Local Authority")
               & (coverage.age_band.isin(["25-49", "50-64"]))
               & coverage.screens_to_target.notna()]

by_band = gap.groupby("age_band").screens_to_target.sum()
total = by_band.sum()

fact("screens_to_target_total", int(total), "screens", "computed", "mi_quarterly_la",
     "Publisher's own per-authority column, summed across England at 31 March 2024")
for band, v in by_band.items():
    fact(f"screens_to_target_{band.replace('-', '_')}", int(v), "screens", "computed", "mi_quarterly_la")

fact("kits_announced", 4_000_000, "kits", "read from source", "nhs_england_press_release",
     "'Nearly 4 million women' — the release does not give a precise figure")
fact("kits_to_gap_ratio", round(4_000_000 / total, 2), "ratio", "computed", "",
     "Kits announced against measured additional screens needed. Different populations: the gap is "
     "measured on ages 25-64, the kits go to non-responders aged 30-65, and a kit posted is not a "
     "screen completed.")

print(by_band.to_string())
print(f"total {total:,.0f} additional screens to reach {TARGET_PCT:.0f}%")
print(f"against ~4,000,000 kits announced -> ratio {4_000_000/total:.2f}")

## 4. The coverage timeline

Every national observation from all three series, with the comparability flag carried through.

The last point is 30 June 2024, from the Sub-ICB file, and it is flagged not comparable. Any chart
built on this table has to respect that flag rather than plotting it as a fall.

In [ ]:
national = coverage[(coverage.geography_level == "National") & (coverage.age_band == "25-64")].copy()

# the Sub-ICB file has no England row, so the national figure for those quarters is the sum of parts
subicb = (coverage[(coverage.source_id == "mi_quarterly_subicb")
                   & (coverage.age_band.isin(["25-49", "50-64"]))]
          .groupby(["series", "source_id", "period_end", "quarter", "comparable",
                    "publication_date"], dropna=False)
          .agg(eligible=("eligible", "sum"), screened=("screened", "sum")).reset_index())
subicb["coverage_pct_computed"] = 100 * subicb.screened / subicb.eligible
subicb["geography_level"] = "National"
subicb["age_band"] = "25-64"
subicb["age_band_source"] = "derived"
subicb["org_name"] = "England"

timeline = (pd.concat([national, subicb], ignore_index=True)
            [["series", "source_id", "period_end", "quarter", "publication_date",
              "coverage_pct_computed", "eligible", "screened",
              "age_band_source", "comparable"]]
            .sort_values(["period_end", "series"])
            .reset_index(drop=True))
timeline["lag_days"] = (pd.to_datetime(timeline.publication_date)
                        - pd.to_datetime(timeline.period_end)).dt.days

last_obs = timeline.period_end.max()
fact("last_public_observation", str(pd.Timestamp(last_obs).date()), "date", "computed", "mi_quarterly_subicb",
     "The most recent period end for which any coverage figure has been published")
fact("days_last_observation_to_announcement",
     (ANNOUNCEMENT - pd.Timestamp(last_obs)).days, "days", "computed")
fact("days_reference_to_announcement", (ANNOUNCEMENT - MAR_2024).days, "days", "computed", "",
     "31 March 2024, the date the headline figure describes, to the announcement")

print(timeline.tail(8).to_string(index=False))
print(f"\nlast observation {pd.Timestamp(last_obs).date()}, "
      f"{(ANNOUNCEMENT - pd.Timestamp(last_obs)).days} days before the announcement")

## 4a. The decade

The annual series runs back to 2013-14, so the timeline can say what the press release does not:
which direction this has been moving. Coverage is a slow-moving stock, and it has been falling.

In [ ]:
decade = (timeline[timeline.series == "annual"]
          .set_index("period_end").coverage_pct_computed.sort_index())
first, last = decade.iloc[0], decade.iloc[-1]
year_on_year = decade.diff().dropna()

fact("national_coverage_first_year", round(first, 4), "%", "computed", "annual_phof",
     f"At {decade.index[0].date()}, the earliest year the annual file carries")
fact("national_coverage_latest_year", round(last, 4), "%", "computed", "annual_phof",
     f"At {decade.index[-1].date()}")
fact("decade_change_pp", round(last - first, 4), "pp", "computed", "annual_phof",
     "Change across the full annual series")
fact("years_of_decline", int((year_on_year < 0).sum()), "years", "computed", "annual_phof",
     f"Of {len(year_on_year)} year-on-year changes in the series")

print(decade.round(4).to_string())
print(f"\n{decade.index[0].date()} to {decade.index[-1].date()}: "
      f"{first:.2f}% to {last:.2f}%, {last - first:+.2f} pp")
print(f"fell in {(year_on_year < 0).sum()} of {len(year_on_year)} years")

## 4b. Do the sources agree with each other?

Three publications describe the same dates. Comparing them is free and it changes what the article
can honestly claim.

Two things come out of it. The annual publication and the quarterly management information give
materially different national figures for the same day — because the annual counts women by where
they live and the quarterly counts them by where they are registered with a GP. And the GOV.UK
quarterly files turn out to agree with the ceased dashboard to within rounding, which means they are
not a second, independent series at all.

In [ ]:
agree = (timeline.pivot_table(index="period_end", columns="series",
                              values="coverage_pct_computed", aggfunc="first")
         .join(timeline.pivot_table(index="period_end", columns="series",
                                    values="eligible", aggfunc="first"),
               rsuffix="_eligible")
         .sort_index())

print(agree.round(4).to_string())

# annual vs quarterly at 31 March 2024: residence base against registration base
ann = timeline[(timeline.series == "annual") & (timeline.period_end == MAR_2024)].iloc[0]
mi_ = timeline[(timeline.source_id == "mi_quarterly_subicb") & (timeline.period_end == MAR_2024)].iloc[0]
base_gap = mi_.coverage_pct_computed - ann.coverage_pct_computed

fact("national_coverage_residence_base", round(ann.coverage_pct_computed, 4), "%", "computed",
     "annual_phof", "Local authority of residence; the accredited annual publication")
fact("national_coverage_registration_base", round(mi_.coverage_pct_computed, 4), "%", "computed",
     "mi_quarterly_subicb", "Sub-ICB of GP registration; management information")
fact("residence_vs_registration_gap_pp", round(base_gap, 4), "pp", "computed", "",
     "Same date, two population bases. The eligible denominators differ by "
     f"{int(ann.eligible - mi_.eligible):,} people.")

# GOV.UK against the ceased dashboard, on the quarters both cover
pair = timeline[timeline.series.isin(["govuk_quarterly", "mi_quarterly"])]
pair = (pair.pivot_table(index="period_end", columns="series", values="coverage_pct_computed")
            .dropna())
pair["difference_pp"] = (pair.govuk_quarterly - pair.mi_quarterly).abs()
worst_pair = pair.difference_pp.max()

fact("govuk_vs_dashboard_max_difference_pp", round(worst_pair, 6), "pp", "computed", "",
     "Across every quarter both publish. Two readings of the same comparison. First: the figures "
     "are effectively identical, so the GOV.UK files republish the same underlying data rather "
     "than providing an independent series. Second: the GOV.UK England figure is published as a "
     "total, while the dashboard figure here is summed from Sub-ICB rows — so their agreement is "
     "evidence that disclosure control does not materially bite at Sub-ICB grain, which is why "
     "that file is used and the GP practice file is not.")

print()
print(f"31 Mar 2024: residence base {ann.coverage_pct_computed:.4f}% vs "
      f"registration base {mi_.coverage_pct_computed:.4f}%  ->  {base_gap:.2f} pp apart")
print(f"eligible denominators differ by {int(ann.eligible - mi_.eligible):,} people")
print()
print("GOV.UK against the ceased dashboard, quarters both cover:")
print(pair.round(6).to_string())

## 5. The publication register

The manifest holds the nine publications this pipeline downloaded. The annual series goes back to
2004-05, and that history is not in any data file — it is a list on the publisher's series page,
which Bronze captured as evidence.

Parsing it here rather than typing it means the cadence figures come from NHS England Digital's own
page, saved and hashed on the day it was read. If someone disputes the 638 days, the evidence is on
disk.

In [ ]:
TITLE_RE = re.compile(r"Cervical [Ss]creening [Pp]rogramme.*?(\d{4})\s*[-–]\s*(\d{2,4})")
DATE_RE  = re.compile(r"^\d{1,2} [A-Z][a-z]+ \d{4}$")

def parse_annual_series(path):
    """Pair each publication title on the series page with the date that follows it."""
    text = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser").get_text("\n")
    rows = [ln.strip() for ln in text.split("\n") if ln.strip()]

    # the latest publication shows as "Published: <date>" rather than a bare date line
    latest = re.search(r"Published:\s*(\d{1,2} [A-Z][a-z]+ \d{4})", "\n".join(rows))

    out, seen = [], set()
    for i, line in enumerate(rows):
        m = TITLE_RE.search(line)
        if not m:
            continue
        start, end = m.group(1), m.group(2)
        collection = f"{start}-{end[-2:]}"
        if collection in seen:
            continue
        date = next((rows[j] for j in range(i + 1, min(i + 6, len(rows)))
                     if DATE_RE.match(rows[j])), None)
        if date is None and latest:
            date = latest.group(1)
        if date:
            seen.add(collection)
            out.append(dict(collection_year=collection,
                            publication_date=pd.to_datetime(date, dayfirst=True),
                            reference_period_end=pd.Timestamp(f"{int(start)+1}-03-31")))
    return pd.DataFrame(out).sort_values("publication_date").reset_index(drop=True)

annual_series = parse_annual_series(EVIDENCE_DIR / "page_nhsd_annual_series.html")
annual_series["gap_days"] = annual_series.publication_date.diff().dt.days

longest_hist = int(annual_series.gap_days.max())
last_annual = annual_series.publication_date.max()
open_gap = (AS_AT - last_annual).days

fact("annual_publications_count", len(annual_series), "publications", "computed", "page_nhsd_annual_series.html")
fact("annual_first_publication", str(annual_series.publication_date.min().date()), "date",
     "computed", "page_nhsd_annual_series.html")
fact("annual_last_publication", str(last_annual.date()), "date", "computed", "page_nhsd_annual_series.html")
fact("annual_longest_historical_gap", longest_hist, "days", "computed", "page_nhsd_annual_series.html",
     "Longest interval between consecutive annual publications before the current one")
fact("annual_open_gap_days", open_gap, "days", "computed", "page_nhsd_annual_series.html",
     f"From {last_annual.date()} to {AS_AT.date()}, still running")
fact("annual_open_gap_ratio", round(open_gap / longest_hist, 2), "ratio", "computed")
fact("days_last_new_period_to_announcement",
     (ANNOUNCEMENT - pd.Timestamp("2025-01-07")).days, "days", "computed", "mi_quarterly_la",
     "7 January 2025, the last publication to add a period nobody had seen before. GOV.UK "
     "published again on 16 July 2025, but that file republished the March 2024 quarter rather "
     "than adding one.")
fact("last_publication_event", "2025-07-16", "date", "computed", "manifest",
     "The most recent publication of any kind in these three series, republished data")

register = pd.concat([
    publications.assign(kind="downloaded")[
        ["series", "source_id", "reference_period_end", "publication_date", "lag_days", "status", "kind"]],
    annual_series.assign(series="annual", source_id="annual_" + annual_series.collection_year,
                         status="not downloaded", kind="series history",
                         lag_days=(annual_series.publication_date
                                   - annual_series.reference_period_end).dt.days)[
        ["series", "source_id", "reference_period_end", "publication_date", "lag_days", "status", "kind"]],
], ignore_index=True).drop_duplicates(
    subset=["series", "reference_period_end", "publication_date"]).sort_values("publication_date")

print(annual_series[["collection_year", "publication_date", "gap_days"]].tail(8).to_string(index=False))
print(f"\n{len(annual_series)} annual publications, longest historical gap {longest_hist} days")
print(f"open gap {open_gap} days ({open_gap/longest_hist:.2f}x the previous record)")

govuk_lags = publications[publications.series == "govuk_quarterly"].lag_days
fact("govuk_lag_min", int(govuk_lags.min()), "days", "computed", "manifest")
fact("govuk_lag_max", int(govuk_lags.max()), "days", "computed", "manifest")

## 6. The facts table

Every number the article uses, named once, with the source it came from and whether it was computed
from data or read off a page. The prose quotes `facts['name']` and never a literal.

In [ ]:
fact("announcement_date", str(ANNOUNCEMENT.date()), "date", "read from source",
     "nhs_england_press_release")
fact("elimination_target_year", 2040, "year", "read from source", "nhs_england_press_release")
fact("first_phase_age_range", "30 to 65", "", "read from source", "nhs_england_press_release")
fact("youscreen_additional_screened", 400_000, "per year", "read from source",
     "nhs_england_press_release", "'about 400,000 each year' from the YouScreen trial")
fact("mi_la_last_quarter",
     coverage[coverage.source_id == "mi_quarterly_la"].quarter.dropna().max(), "", "computed")
fact("mi_subicb_last_quarter",
     coverage[coverage.source_id == "mi_quarterly_subicb"].quarter.dropna().max(), "", "computed")

facts_df = pd.DataFrame(facts).drop_duplicates(subset="name", keep="last").reset_index(drop=True)
print(f"{len(facts_df)} facts")
facts_df

## 7. Checks

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append(dict(check=name, passed=bool(ok), detail=str(detail)))

f = facts_df.set_index("name").value

check("headline rounds to the published 68.8%", round(float(f.headline_coverage_pct), 1) == 68.8,
      f.headline_coverage_pct)
check("no local authority reaches the target", int(f.la_at_or_above_target) == 0,
      f.la_at_or_above_target)
check("distribution covers every authority in the annual file",
      len(la) == int(f.la_count_annual))
check("quarterly lists two more authorities than annual",
      int(f.la_count_quarterly) - int(f.la_count_annual) == 2)
check("20 annual publications parsed from the series page",
      int(f.annual_publications_count) == 20, f.annual_publications_count)
check("open gap exceeds the longest historical gap",
      int(f.annual_open_gap_days) > int(f.annual_longest_historical_gap),
      f"{f.annual_open_gap_days} vs {f.annual_longest_historical_gap}")
check("last public observation is 30 June 2024",
      str(f.last_public_observation) == "2024-06-30", f.last_public_observation)
check("timeline's final point is flagged not comparable",
      not bool(timeline.sort_values("period_end").iloc[-1].comparable))
# Suppression check: a published England total against the sum of published Sub-ICB rows.
# If small-number suppression were biting at Sub-ICB grain, the sum would fall short.
check("Sub-ICB sums reconcile to an independently published England total",
      float(f.govuk_vs_dashboard_max_difference_pp) < 0.001,
      f"worst {f.govuk_vs_dashboard_max_difference_pp} pp")
check("every fact names a basis", facts_df.basis.isin(["computed", "read from source"]).all())
check("every read-from-source fact names a source",
      (facts_df[facts_df.basis == "read from source"].source != "").all())

results = pd.DataFrame(checks)
print(f"{results.passed.sum()}/{len(results)} checks passed")
if not results.passed.all():
    print(results[~results.passed].to_string(index=False))
assert results.passed.all(), "A Gold check failed."

## 8. Save and summarise

In [ ]:
la.to_parquet(GOLD_DIR / "la_coverage_2024_03_31.parquet", index=False)
la.to_csv(GOLD_DIR / "la_coverage_2024_03_31.csv", index=False)
timeline.to_parquet(GOLD_DIR / "coverage_timeline.parquet", index=False)
register.to_parquet(GOLD_DIR / "publication_register.parquet", index=False)
register.to_csv(GOLD_DIR / "publication_register.csv", index=False)
annual_series.to_csv(GOLD_DIR / "annual_publication_cadence.csv", index=False)
facts_df.to_csv(GOLD_DIR / "facts.csv", index=False)

for name, df in [("la_coverage_2024_03_31", la), ("coverage_timeline", timeline),
                 ("publication_register", register), ("annual_publication_cadence", annual_series),
                 ("facts", facts_df)]:
    print(f"{name:<28} {len(df):>5,} rows")

print()
print(facts_df[facts_df.name.isin([
    "headline_coverage_pct", "la_at_or_above_target", "la_spread_pp",
    "screens_to_target_total", "kits_to_gap_ratio",
    "days_reference_to_announcement", "annual_open_gap_days", "annual_longest_historical_gap",
])][["name", "value", "unit"]].to_string(index=False))